# Weather Data Demo

Demonstrates fetching AWH-relevant climate data (RH, temperature, solar irradiance)
for any location in the world using the Open-Meteo API.

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root to path

from src.data.weather import WeatherClient

client = WeatherClient()

## Fetch historical data by coordinates

In [ ]:
# Phoenix, AZ — a hot/arid test case
df = client.get_historical(
    latitude=33.45,
    longitude=-112.07,
    start='2023-06-01',
    end='2023-06-30',
)
df.head()

## Fetch historical data by place name

In [ ]:
# High-humidity tropical site
df_singapore = client.get_historical_by_name(
    'Singapore',
    start='2023-01-01',
    end='2023-12-31',
)
print(df_singapore['location_name'].iloc[0])
df_singapore[['temperature_2m', 'relative_humidity_2m', 'shortwave_radiation']].describe()

## Compare AWH potential across sites

In [ ]:
import matplotlib.pyplot as plt

sites = {
    'Phoenix, AZ': (33.45, -112.07),
    'Singapore':   (1.35,   103.82),
    'Riyadh':      (24.69,   46.72),
    'London':      (51.51,   -0.13),
    'Atacama':     (-23.87, -69.85),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (lat, lon) in sites.items():
    d = client.get_historical(lat, lon, '2023-01-01', '2023-12-31')
    monthly = d[['relative_humidity_2m', 'shortwave_radiation']].resample('ME').mean()
    axes[0].plot(monthly.index, monthly['relative_humidity_2m'], label=name)
    axes[1].plot(monthly.index, monthly['shortwave_radiation'], label=name)

axes[0].set_title('Monthly Mean Relative Humidity (%)')
axes[0].set_ylabel('RH (%)')
axes[0].legend(fontsize=8)

axes[1].set_title('Monthly Mean GHI (W/m²)')
axes[1].set_ylabel('GHI (W/m²)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Monthly climate summary

In [ ]:
summary = client.get_climate_summary(
    latitude=33.45, longitude=-112.07,
    start='2023-01-01', end='2023-12-31',
    freq='ME'
)
# Show mean RH and GHI by month
summary[['relative_humidity_2m', 'shortwave_radiation']].xs('mean', axis=1, level=1)